In [19]:
import re
import json
import time
import hashlib
from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI
from docx import Document

load_dotenv()

# -----------------------------
# 설정
# -----------------------------
MODEL_NAME = "gpt-4.1-mini"
INPUT_FILE = "fia_2026_f1_regulations_-_section_c_technical_-_iss_16_-_2026-02-27.docx"

OUTPUT_DIR = Path("output_qa_pipeline_docx_final_clean")
OUTPUT_DIR.mkdir(exist_ok=True)

ARTICLE_JSON_PATH = OUTPUT_DIR / "articles.json"
RAW_QA_PATH = OUTPUT_DIR / "raw_qa.json"
DEDUP_QA_PATH = OUTPUT_DIR / "dedup_qa.json"
FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

TARGET_QA_COUNT = 3000
RAW_TARGET_QA_COUNT = 3600

GENERATION_CONFIG = {
    "concept": 4,
    "summary": 4,
    "numeric": 2,
    "misconception": 4,
    "comparison": 2
}

SLEEP_SECONDS = 0.1
MAX_RETRIES = 3

IMPORTANT_SUBARTICLES = {
    # C1~C2
    "C1.1.1", "C1.2.1", "C1.2.2", "C1.3.1", "C1.5.1",
    "C2.1.1", "C2.1.2", "C2.1.3", "C2.2.1", "C2.3.1", "C2.3.2", "C2.3.3",

    # C3 general
    "C3.1.1",
    "C3.2.1", "C3.2.2", "C3.2.3", "C3.2.4", "C3.2.5", "C3.2.6", "C3.2.7",
    "C3.3.1", "C3.3.2", "C3.3.3", "C3.3.4",
    "C3.4.1", "C3.4.2", "C3.4.3", "C3.4.4",

    # Floor / Plank
    "C3.5.1", "C3.5.4", "C3.5.5", "C3.5.6", "C3.5.7", "C3.5.8", "C3.5.9",
    "C3.5.10", "C3.5.11", "C3.5.12", "C3.5.13", "C3.5.14",
    "C3.6.1", "C3.6.2", "C3.6.3", "C3.6.4",

    # Front / Rear bodywork
    "C3.7.1", "C3.7.2", "C3.7.3", "C3.7.5", "C3.7.7",
    "C3.8.1", "C3.8.2", "C3.8.3",
    "C3.9.1", "C3.9.2",

    # Front wing
    "C3.10.1", "C3.10.2", "C3.10.3", "C3.10.5", "C3.10.6",
    "C3.10.7", "C3.10.8", "C3.10.9", "C3.10.10", "C3.10.11", "C3.10.12",

    # Rear wing
    "C3.11.1", "C3.11.2", "C3.11.3", "C3.11.4", "C3.11.5", "C3.11.6", "C3.11.7", "C3.11.8",

    # Final assembly / others
    "C3.12.1", "C3.12.2", "C3.12.3", "C3.12.4", "C3.12.5",
    "C3.13.1", "C3.13.4", "C3.14.1", "C3.14.2", "C3.14.3", "C3.14.4",

    # Other major topics
    "C4.1", "C4.2", "C4.5",
    "C5.2", "C5.17", "C5.18", "C5.19", "C5.23",
    "C6.4", "C6.5",
    "C8.5", "C8.6", "C8.9", "C8.10",
    "C10.8", "C10.9",
    "C11.6",
    "C12.2", "C12.5", "C12.8",
    "C13.2", "C13.5", "C13.6", "C13.7",
    "C14.4", "C14.5", "C14.6",
    "C16.2", "C16.3", "C16.5",
    "C17.2", "C17.3", "C17.4", "C17.5", "C17.6", "C17.7",
    "C18.2", "C18.3", "C18.4", "C18.5", "C18.6",
}

client = OpenAI()


# -----------------------------
# 1) DOCX 읽기
# -----------------------------
def read_docx(path: str) -> str:
    doc = Document(path)
    lines = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            lines.append(text)
    return "\n".join(lines)


# -----------------------------
# 2) 텍스트 정리 / 보정
# -----------------------------
def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = text.replace("−", "-")
    text = text.replace("–", "-")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def keep_main_body(text: str) -> str:
    marker = "ARTICLE C1: GENERAL PRINCIPLES"
    idx = text.find(marker)
    if idx == -1:
        raise ValueError("본문 시작점 'ARTICLE C1: GENERAL PRINCIPLES'를 찾지 못했습니다.")
    return text[idx:]


def fix_common_ocr_errors(text: str) -> str:
    replacements = {
        "ARTICLE CG:": "ARTICLE C9:",
        "ARTICLE CG ": "ARTICLE C9 ",
        "ARTICLE CG\n": "ARTICLE C9\n",
        "C3.G\t": "C3.9\t",
        "C3.G ": "C3.9 ",
        "C3.G\n": "C3.9\n",
        "C3.5.G ": "C3.5.9 ",
        "C3.5.G\n": "C3.5.9\n",
        "C3.10.G ": "C3.10.9 ",
        "C3.10.G\n": "C3.10.9\n",
    }
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    text = re.sub(r"\b(C\d+)\.G\b", r"\1.9", text)
    text = re.sub(r"\b(C\d+\.\d+)\.G\b", r"\1.9", text)
    return text


def remove_table_of_contents_noise(text: str) -> str:
    lines = text.split("\n")
    cleaned = []

    noise_patterns = [
        r"^SECTION C: TECHNICAL REGULATIONS$",
        r"^27 February 2026$",
        r"^Issue 16$",
        r"^2026 Formula 1: Technical Regulations$",
        r"^©2026 Fédération Internationale de l’Automobile$",
        r"^C\d+$",
    ]

    for line in lines:
        s = line.strip()
        if not s:
            cleaned.append("")
            continue
        if any(re.match(p, s) for p in noise_patterns):
            continue
        cleaned.append(s)

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# -----------------------------
# 3) 본문에서 2단계 제목 추출
# -----------------------------
def extract_level2_title_map_from_main_body(text: str) -> Dict[str, str]:
    """
    본문에서 'C1.1 Formula One World Championship' 같은
    2단계 조항 제목 라인을 직접 추출
    """
    text = keep_main_body(text)
    text = remove_table_of_contents_noise(text)

    title_map = {}

    pattern = re.compile(r"(?m)^(C\d+\.\d+)\s+([^\n]+)$")

    for m in pattern.finditer(text):
        article_id = m.group(1).strip()
        title = m.group(2).strip()

        if re.match(r"^\.\d+", title):
            continue
        if re.match(r"^C\d+\.\d+(?:\.\d+)?", title):
            continue
        if len(title) > 120:
            continue

        title_map[article_id] = title

    return title_map


# -----------------------------
# 4) 조문 분리 보조
# -----------------------------
def article_sort_key(article_id: str):
    nums = article_id[1:].split(".")
    return tuple(int(n) for n in nums)


def clean_article_title(article_id: str, body: str, title_map: Dict[str, str]) -> str:
    # 1순위: 2단계 제목 맵
    if article_id in title_map and title_map[article_id]:
        return title_map[article_id]

    # 2순위: 본문에서 제목처럼 보이는 짧은 줄
    lines = [line.strip() for line in body.split("\n") if line.strip()]

    for line in lines[:8]:
        candidate = re.sub(rf"^{re.escape(article_id)}\s*", "", line).strip()

        if not candidate:
            continue
        if re.match(r"^\.\d+", candidate):
            continue
        if re.match(r"^C\d+\.\d+(?:\.\d+)?", candidate):
            continue
        if len(candidate) > 80:
            continue

        return candidate

    return article_id


def extract_blocks(text: str, pattern: re.Pattern, title_map: Dict[str, str]) -> List[Dict[str, str]]:
    matches = list(pattern.finditer(text))
    blocks = []

    if not matches:
        return blocks

    for i, match in enumerate(matches):
        article_id = match.group(1).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()

        if len(body) < 40:
            continue

        title = clean_article_title(article_id, body, title_map)

        blocks.append({
            "article_id": article_id,
            "article_title": title,
            "article_text": body
        })

    return blocks


# -----------------------------
# 5) 혼합형 조문 분리
# -----------------------------
def split_articles(text: str) -> List[Dict[str, Any]]:
    raw_text = normalize_text(text)
    raw_text = fix_common_ocr_errors(raw_text)

    # 목차가 아니라 본문에서 2단계 제목 직접 추출
    title_map = extract_level2_title_map_from_main_body(raw_text)

    text = keep_main_body(raw_text)
    text = remove_table_of_contents_noise(text)

    pattern_lvl2 = re.compile(r"(?m)^(C\d+\.\d+)\s*(.*)$")
    pattern_lvl3 = re.compile(r"(?m)^(C\d+\.\d+\.\d+)\s*(.*)$")

    lvl2_blocks = extract_blocks(text, pattern_lvl2, title_map)
    lvl3_blocks = extract_blocks(text, pattern_lvl3, title_map)

    merged = {b["article_id"]: b for b in lvl2_blocks}

    for b in lvl3_blocks:
        if b["article_id"] in IMPORTANT_SUBARTICLES:
            merged[b["article_id"]] = b

    articles = list(merged.values())
    articles.sort(key=lambda x: article_sort_key(x["article_id"]))

    return articles


# -----------------------------
# 6) Q/A 생성
# -----------------------------
def extract_json_array(text: str) -> str:
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def build_single_call_prompt(article: Dict[str, Any], generation_config: Dict[str, int]) -> str:
    question_type_desc = {
        "concept": "개념 설명형",
        "summary": "규정 요약형",
        "numeric": "수치/기준형",
        "misconception": "오해 교정형",
        "comparison": "비교형"
    }

    type_lines = []
    for qtype, n in generation_config.items():
        if n > 0:
            type_lines.append(f"- {qtype}: {n}개 ({question_type_desc[qtype]})")

    joined_types = "\n".join(type_lines)

    prompt = f"""
너는 F1 입문자용 기술규정 설명 챗봇의 파인튜닝 데이터를 만드는 역할이야.

아래 FIA F1 기술규정 조문을 읽고, 조문 내용만을 바탕으로 한국어 Q/A를 생성해라.

[조문 정보]
- 조항 ID: {article['article_id']}
- 조항 제목: {article['article_title']}

[조문 원문]
{article['article_text']}

[생성 개수]
{joined_types}

[생성 규칙]
- 조문 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 2~4문장
- 숫자, 단위(mm, ms, 개수, 각도)는 조문 기준 그대로 유지
- 규정 원문에 없는 내용 단정 금지
- 너무 직역체로 쓰지 말 것
- 질문끼리 의미 중복이 크지 않게 작성
- 오해 교정형은 왜 틀렸는지도 설명할 것
- 비교형은 같은 조문 안에서 비교 가능한 대상이 있을 때만 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것

[출력 형식]
반드시 JSON 배열만 출력:
[
  {{
    "article_id": "{article['article_id']}",
    "article_title": "{article['article_title']}",
    "question_type": "concept|summary|numeric|misconception|comparison",
    "question": "...",
    "answer": "..."
  }}
]
""".strip()

    return prompt


def generate_qa_for_article(article: Dict[str, Any], generation_config: Dict[str, int]) -> List[Dict[str, Any]]:
    prompt = build_single_call_prompt(article, generation_config)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )
            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                cleaned = extract_json_array(text)
                data = json.loads(cleaned)

            validated = []
            for item in data:
                if not isinstance(item, dict):
                    continue
                if "question" not in item or "answer" not in item:
                    continue
                item["article_id"] = article["article_id"]
                item["article_title"] = article["article_title"]
                validated.append(item)

            return validated

        except Exception as e:
            print(f"[재시도 {attempt+1}/{MAX_RETRIES}] {article['article_id']} 생성 실패: {e}")
            time.sleep(1.5)

    return []


# -----------------------------
# 7) 중복 제거
# -----------------------------
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def make_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        q_norm = normalize_question(item["question"])
        a_norm = re.sub(r"\s+", " ", item["answer"].strip().lower())
        key = make_hash(q_norm + "||" + a_norm)

        if key in seen:
            continue
        seen.add(key)
        deduped.append(item)

    return deduped


# -----------------------------
# 8) 저장
# -----------------------------
def to_finetune_messages(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "messages": [
            {
                "role": "system",
                "content": "너는 F1 입문자용 기술규정 설명 챗봇이야. 항상 한국어로 쉽고 친절하게 설명해. 규정에 없는 내용은 단정하지 말고, 숫자와 기준은 정확히 설명해."
            },
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }


def save_json(path: Path, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# -----------------------------
# 9) 실행
# -----------------------------
def main():
    print("[1/5] DOCX 로드")
    full_text = read_docx(INPUT_FILE)

    print("[2/5] 조문 분리")
    articles = split_articles(full_text)
    print(f"분리된 조문 수: {len(articles)}")
    print(f"조문당 요청 생성 수: {sum(GENERATION_CONFIG.values())}")
    print(f"예상 원본 최대 Q/A 수: {len(articles) * sum(GENERATION_CONFIG.values())}")

    save_json(ARTICLE_JSON_PATH, articles)

    print("\n[조문 샘플 25개]")
    for a in articles[:25]:
        print(f"{a['article_id']} | {a['article_title']}")

    print("\n[3/5] Q/A 생성")
    raw_qa = []

    for idx, article in enumerate(articles, start=1):
        if len(raw_qa) >= RAW_TARGET_QA_COUNT:
            print(f"원본 목표 개수 {RAW_TARGET_QA_COUNT}개에 도달해서 생성 중단")
            break

        print(f"  - ({idx}/{len(articles)}) {article['article_id']} {article['article_title']}")
        qa_items = generate_qa_for_article(article, GENERATION_CONFIG)

        remaining = RAW_TARGET_QA_COUNT - len(raw_qa)
        if remaining <= 0:
            break

        raw_qa.extend(qa_items[:remaining])

        if idx % 10 == 0:
            save_json(RAW_QA_PATH, raw_qa)

        print(f"    현재 누적 원본 Q/A 수: {len(raw_qa)} / {RAW_TARGET_QA_COUNT}")
        time.sleep(SLEEP_SECONDS)

    save_json(RAW_QA_PATH, raw_qa)
    print(f"생성된 원본 Q/A 수: {len(raw_qa)}")

    print("[4/5] 중복 제거")
    deduped = deduplicate_qa(raw_qa)
    deduped = deduped[:TARGET_QA_COUNT]

    save_json(DEDUP_QA_PATH, deduped)
    print(f"중복 제거 후 최종 Q/A 수: {len(deduped)}")

    print("[5/5] 파인튜닝 JSONL 저장")
    finetune_rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, finetune_rows)

    print("\n완료")
    print(f"- 조문 저장: {ARTICLE_JSON_PATH}")
    print(f"- 원본 Q/A: {RAW_QA_PATH}")
    print(f"- 중복 제거 Q/A: {DEDUP_QA_PATH}")
    print(f"- 파인튜닝 JSONL: {FINETUNE_JSONL_PATH}")


if __name__ == "__main__":
    main()

[1/5] DOCX 로드
[2/5] 조문 분리
분리된 조문 수: 245
조문당 요청 생성 수: 16
예상 원본 최대 Q/A 수: 3920

[조문 샘플 25개]
C1.1 | Formula One World Championship
C1.1.1 | C1.1.1
C1.2 | Regulatory Framework
C1.2.1 | C1.2.1
C1.2.2 | C1.2.2
C1.3 | Interpretation of and amendments to these Technical Regulations
C1.3.1 | C1.3.1
C1.4 | Dangerous construction
C1.5 | Compliance with the regulations
C1.6 | New systems or technologies
C1.7 | Duty of Competitor and PU Manufacturer
C2.1 | Coordinate systems and conventions
C2.1.1 | Car Coordinate System
C2.1.2 | Further conventions
C2.1.3 | Wheel Coordinate System
C2.2 | Principal Planes
C2.3 | Fundamental Dimensions
C2.3.1 | Width
C2.3.2 | Height
C2.3.3 | Wheelbase
C2.4 | Reference Volumes and Surfaces
C2.5 | Precision of Numerical Values
C3.1 | Aerodynamic Components or Bodywork
C3.2 | General Principles
C3.2.1 | Objectives of Article C3

[3/5] Q/A 생성
  - (1/245) C1.1 Formula One World Championship
    현재 누적 원본 Q/A 수: 16 / 3600
  - (2/245) C1.1.1 C1.1.1
    현재 누적 원본 Q/A 수: 32 / 

In [8]:
import re
import json
import time
import math
import random
import hashlib
from pathlib import Path
from typing import List, Dict, Any
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# 설정
# -----------------------------
MODEL_NAME = "gpt-4.1-mini"
INPUT_FILE = "fia_2026_f1_regulations_-_section_c_technical_-_iss_16_-_2026-02-27.md"

OUTPUT_DIR = Path("output_qa_pipeline_md")
OUTPUT_DIR.mkdir(exist_ok=True)

ARTICLE_JSON_PATH = OUTPUT_DIR / "articles.json"
RAW_QA_PATH = OUTPUT_DIR / "raw_qa.json"
DEDUP_QA_PATH = OUTPUT_DIR / "dedup_qa.json"
FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

# 테스트용
TEST_MODE = False
TEST_ARTICLE_LIMIT = 10

# 목표 개수
TARGET_FINAL_QA = 5000

# 조문당 기본 생성 개수
GENERATION_CONFIG = {
    "concept": 3,
    "summary": 2,
    "numeric": 1,
    "misconception": 2,
    "comparison": 1
}

SLEEP_SECONDS = 0.2
MAX_RETRIES = 3
MAX_PASSES = 20  # 전체 조문을 최대 몇 바퀴 돌지

IMPORTANT_SUBARTICLES = {
    "C1.1.1", "C1.2.1", "C1.2.2", "C1.3.1", "C1.5.1",
    "C2.1.1", "C2.1.2", "C2.1.3", "C2.2.1", "C2.3.1", "C2.3.2", "C2.3.3",
    "C3.1.1",
    "C3.2.1", "C3.2.2", "C3.2.3", "C3.2.4", "C3.2.5", "C3.2.6", "C3.2.7",
    "C3.3.1", "C3.3.2", "C3.3.3", "C3.3.4",
    "C3.4.1", "C3.4.2", "C3.4.3", "C3.4.4",
    "C3.5.1", "C3.5.4", "C3.5.5", "C3.5.6", "C3.5.7", "C3.5.8", "C3.5.9",
    "C3.5.10", "C3.5.11", "C3.5.12", "C3.5.13", "C3.5.14",
    "C3.6.1", "C3.6.2", "C3.6.3", "C3.6.4",
    "C3.7.1", "C3.7.2", "C3.7.3", "C3.7.5", "C3.7.7",
    "C3.8.1", "C3.8.2", "C3.8.3",
    "C3.9.1", "C3.9.2",
    "C3.10.1", "C3.10.2", "C3.10.3", "C3.10.5", "C3.10.6",
    "C3.10.7", "C3.10.8", "C3.10.9", "C3.10.10", "C3.10.11", "C3.10.12",
    "C3.11.1", "C3.11.2", "C3.11.3", "C3.11.4", "C3.11.5", "C3.11.6", "C3.11.7", "C3.11.8",
    "C3.12.1", "C3.12.2", "C3.12.3", "C3.12.4", "C3.12.5",
    "C3.13.1", "C3.13.4", "C3.14.1", "C3.14.2", "C3.14.3", "C3.14.4",
    "C4.1", "C4.2", "C4.5",
    "C5.2", "C5.17", "C5.18", "C5.19", "C5.23",
    "C6.4", "C6.5",
    "C8.5", "C8.6", "C8.9", "C8.10",
    "C10.8", "C10.9",
    "C11.6",
    "C12.2", "C12.5", "C12.8",
    "C13.2", "C13.5", "C13.6", "C13.7",
    "C14.4", "C14.5", "C14.6",
    "C16.2", "C16.3", "C16.5",
    "C17.2", "C17.3", "C17.4", "C17.5", "C17.6", "C17.7",
    "C18.2", "C18.3", "C18.4", "C18.5", "C18.6",
}

client = OpenAI()


# -----------------------------
# 1) MD 읽기
# -----------------------------
def read_md(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


# -----------------------------
# 2) 텍스트 정리 / 보정
# -----------------------------
def normalize_text(text: str) -> str:
    text = text.replace("\u00ad", "")
    text = text.replace("\xa0", " ")
    text = text.replace("−", "-")
    text = text.replace("–", "-")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def keep_main_body(text: str) -> str:
    marker = "ARTICLE C1: GENERAL PRINCIPLES"
    idx = text.find(marker)
    if idx == -1:
        raise ValueError("본문 시작점 'ARTICLE C1: GENERAL PRINCIPLES'를 찾지 못했습니다.")
    return text[idx:]


def fix_common_ocr_errors(text: str) -> str:
    replacements = {
        "ARTICLE CG:": "ARTICLE C9:",
        "ARTICLE CG ": "ARTICLE C9 ",
        "ARTICLE CG\n": "ARTICLE C9\n",
        "C3.G\t": "C3.9\t",
        "C3.G ": "C3.9 ",
        "C3.G\n": "C3.9\n",
        "C3.5.G ": "C3.5.9 ",
        "C3.5.G\n": "C3.5.9\n",
        "C3.10.G ": "C3.10.9 ",
        "C3.10.G\n": "C3.10.9\n",
    }
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    text = re.sub(r"\b(C\d+)\.G\b", r"\1.9", text)
    text = re.sub(r"\b(C\d+\.\d+)\.G\b", r"\1.9", text)
    return text


def remove_markdown_noise(text: str) -> str:
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        s = line.strip()

        if not s:
            cleaned.append("")
            continue

        s = re.sub(r"^#{1,6}\s*", "", s)
        s = s.replace("**", "")
        cleaned.append(s)

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def remove_table_of_contents_noise(text: str) -> str:
    lines = text.split("\n")
    cleaned = []

    noise_patterns = [
        r"^SECTION C: TECHNICAL REGULATIONS$",
        r"^27 February 2026$",
        r"^Issue 16$",
        r"^2026 Formula 1: Technical Regulations$",
        r"^©2026 Fédération Internationale de l’Automobile$",
        r"^C\d+$",
    ]

    for line in lines:
        s = line.strip()
        if not s:
            cleaned.append("")
            continue
        if any(re.match(p, s) for p in noise_patterns):
            continue
        cleaned.append(s)

    text = "\n".join(cleaned)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# -----------------------------
# 3) 본문에서 2단계 제목 추출
# -----------------------------
def extract_level2_title_map_from_main_body(text: str) -> Dict[str, str]:
    text = keep_main_body(text)
    text = remove_markdown_noise(text)
    text = remove_table_of_contents_noise(text)

    title_map = {}
    pattern = re.compile(r"(?m)^(C\d+\.\d+)\s+([^\n]+)$")

    for m in pattern.finditer(text):
        article_id = m.group(1).strip()
        title = m.group(2).strip()

        if re.match(r"^\.\d+", title):
            continue
        if re.match(r"^C\d+\.\d+(?:\.\d+)?", title):
            continue
        if len(title) > 120:
            continue

        title_map[article_id] = title

    return title_map


# -----------------------------
# 4) 조문 분리 보조
# -----------------------------
def article_sort_key(article_id: str):
    nums = article_id[1:].split(".")
    return tuple(int(n) for n in nums)


def clean_article_title(article_id: str, body: str, title_map: Dict[str, str]) -> str:
    if article_id in title_map and title_map[article_id]:
        return title_map[article_id]

    lines = [line.strip() for line in body.split("\n") if line.strip()]

    for line in lines[:8]:
        candidate = re.sub(rf"^{re.escape(article_id)}\s*", "", line).strip()

        if not candidate:
            continue
        if re.match(r"^\.\d+", candidate):
            continue
        if re.match(r"^C\d+\.\d+(?:\.\d+)?", candidate):
            continue
        if len(candidate) > 80:
            continue

        return candidate

    return article_id


def extract_blocks(text: str, pattern: re.Pattern, title_map: Dict[str, str]) -> List[Dict[str, str]]:
    matches = list(pattern.finditer(text))
    blocks = []

    if not matches:
        return blocks

    for i, match in enumerate(matches):
        article_id = match.group(1).strip()
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()

        if len(body) < 40:
            continue

        title = clean_article_title(article_id, body, title_map)

        blocks.append({
            "article_id": article_id,
            "article_title": title,
            "article_text": body
        })

    return blocks


# -----------------------------
# 5) 혼합형 조문 분리
# -----------------------------
def split_articles(text: str) -> List[Dict[str, Any]]:
    raw_text = normalize_text(text)
    raw_text = remove_markdown_noise(raw_text)
    raw_text = fix_common_ocr_errors(raw_text)

    title_map = extract_level2_title_map_from_main_body(raw_text)

    text = keep_main_body(raw_text)
    text = remove_table_of_contents_noise(text)

    pattern_lvl2 = re.compile(r"(?m)^(C\d+\.\d+)\s*(.*)$")
    pattern_lvl3 = re.compile(r"(?m)^(C\d+\.\d+\.\d+)\s*(.*)$")

    lvl2_blocks = extract_blocks(text, pattern_lvl2, title_map)
    lvl3_blocks = extract_blocks(text, pattern_lvl3, title_map)

    merged = {b["article_id"]: b for b in lvl2_blocks}

    for b in lvl3_blocks:
        if b["article_id"] in IMPORTANT_SUBARTICLES:
            merged[b["article_id"]] = b

    articles = list(merged.values())
    articles.sort(key=lambda x: article_sort_key(x["article_id"]))

    return articles


# -----------------------------
# 6) Q/A 생성
# -----------------------------
def extract_json_array(text: str) -> str:
    match = re.search(r"\[\s*{.*?}\s*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def get_question_type_desc() -> Dict[str, str]:
    return {
        "concept": "개념 설명형",
        "summary": "규정 요약형",
        "numeric": "수치/기준형",
        "misconception": "오해 교정형",
        "comparison": "비교형"
    }


def build_single_call_prompt(
    article: Dict[str, Any],
    generation_config: Dict[str, int],
    pass_index: int = 1,
    seed_hint: str = ""
) -> str:
    question_type_desc = get_question_type_desc()

    type_lines = []
    for qtype, n in generation_config.items():
        if n > 0:
            type_lines.append(f"- {qtype}: {n}개 ({question_type_desc[qtype]})")

    joined_types = "\n".join(type_lines)

    diversity_instruction = [
        "질문 시작 표현을 반복하지 말 것",
        "같은 의미를 단어만 바꿔서 반복하지 말 것",
        "입문자가 실제로 할 법한 질문으로 작성할 것",
        "이전 패스와 겹치지 않게 새로운 각도로 질문할 것",
    ]

    prompt = f"""
너는 F1 입문자용 기술규정 설명 챗봇의 파인튜닝 데이터를 만드는 역할이야.

아래 FIA F1 기술규정 조문을 읽고, 조문 내용만을 바탕으로 한국어 Q/A를 생성해라.

[조문 정보]
- 조항 ID: {article['article_id']}
- 조항 제목: {article['article_title']}

[조문 원문]
{article['article_text']}

[생성 개수]
{joined_types}

[현재 생성 회차]
- pass_index: {pass_index}
- variation_hint: {seed_hint}

[생성 규칙]
- 조문 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 2~4문장
- 숫자, 단위(mm, ms, 개수, 각도)는 조문 기준 그대로 유지
- 규정 원문에 없는 내용 단정 금지
- 질문끼리 의미 중복이 크지 않게 작성
- 오해 교정형은 왜 틀렸는지도 설명할 것
- 비교형은 같은 조문 안에서 비교 가능한 대상이 있을 때만 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 조항 번호만 언급하지 말고, 내용을 풀어서 설명할 것
- 아래 유형을 고루 섞기:
1. 개념형
2. 요약형
3. 수치/기준형
4. 오해 교정형

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()

    return prompt


def validate_qa_items(data: Any, article: Dict[str, Any]) -> List[Dict[str, Any]]:
    validated = []

    if not isinstance(data, list):
        return validated

    for item in data:
        if not isinstance(item, dict):
            continue

        question = item.get("question", "")
        answer = item.get("answer", "")

        if not isinstance(question, str) or not isinstance(answer, str):
            continue

        question = question.strip()
        answer = answer.strip()

        if not question or not answer:
            continue

        validated.append({
            "question": question,
            "answer": answer,
            "article_id": article["article_id"],
            "article_title": article["article_title"]
        })

    return validated


def generate_qa_for_article(
    article: Dict[str, Any],
    generation_config: Dict[str, int],
    pass_index: int = 1
) -> List[Dict[str, Any]]:
    seed_hint = f"{article['article_id']}-{pass_index}-{random.randint(1000, 9999)}"
    prompt = build_single_call_prompt(article, generation_config, pass_index, seed_hint)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )
            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                cleaned = extract_json_array(text)
                data = json.loads(cleaned)

            validated = validate_qa_items(data, article)
            if validated:
                return validated

        except Exception as e:
            print(f"[재시도 {attempt+1}/{MAX_RETRIES}] {article['article_id']} 생성 실패: {e}")
            time.sleep(1.5)

    return []


# -----------------------------
# 7) 중복 제거
# -----------------------------
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def normalize_answer(a: str) -> str:
    a = a.strip().lower()
    a = re.sub(r"\s+", " ", a)
    return a


def make_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        q_norm = normalize_question(item["question"])
        a_norm = normalize_answer(item["answer"])
        key = make_hash(q_norm + "||" + a_norm)

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped


# -----------------------------
# 8) 저장
# -----------------------------
def to_finetune_messages(item: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }


def save_json(path: Path, data: Any) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def save_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


# -----------------------------
# 9) 목표 개수까지 생성
# -----------------------------
def estimate_articles_needed(target_final: int, per_article_raw: int, dedup_survival_rate: float = 0.8) -> int:
    expected_final_per_article = max(1, int(per_article_raw * dedup_survival_rate))
    return math.ceil(target_final / expected_final_per_article)


def generate_until_target(
    articles: List[Dict[str, Any]],
    generation_config: Dict[str, int],
    target_final_qa: int
) -> List[Dict[str, Any]]:
    raw_qa: List[Dict[str, Any]] = []
    pass_index = 1

    while pass_index <= MAX_PASSES:
        print(f"\n===== PASS {pass_index}/{MAX_PASSES} 시작 =====")

        random.shuffle(articles)

        for idx, article in enumerate(articles, start=1):
            deduped_so_far = deduplicate_qa(raw_qa)

            if len(deduped_so_far) >= target_final_qa:
                print("목표 개수 도달. 생성 종료.")
                return raw_qa

            print(
                f"  - PASS {pass_index} | ({idx}/{len(articles)}) "
                f"{article['article_id']} {article['article_title']}"
            )

            qa_items = generate_qa_for_article(article, generation_config, pass_index=pass_index)
            raw_qa.extend(qa_items)

            if len(raw_qa) % 100 == 0 or idx % 10 == 0:
                save_json(RAW_QA_PATH, raw_qa)
                dedup_snapshot = deduplicate_qa(raw_qa)
                save_json(DEDUP_QA_PATH, dedup_snapshot)
                print(
                    f"    누적 원본: {len(raw_qa)} | "
                    f"누적 중복제거: {len(dedup_snapshot)} / 목표 {target_final_qa}"
                )

            time.sleep(SLEEP_SECONDS)

        pass_index += 1

    print("최대 PASS 수에 도달했습니다.")
    return raw_qa


# -----------------------------
# 10) 실행
# -----------------------------
def main():
    print("[1/5] MD 로드")
    full_text = read_md(INPUT_FILE)

    print("[2/5] 조문 분리")
    articles = split_articles(full_text)
    print(f"전체 분리 조문 수: {len(articles)}")

    if TEST_MODE:
        articles = articles[:TEST_ARTICLE_LIMIT]
        print(f"테스트 조문 수: {len(articles)}")
    else:
        print(f"실사용 조문 수: {len(articles)}")

    per_article_raw = sum(GENERATION_CONFIG.values())
    estimated_needed = estimate_articles_needed(
        target_final=TARGET_FINAL_QA,
        per_article_raw=per_article_raw,
        dedup_survival_rate=0.8
    )

    print(f"조문당 요청 생성 수: {per_article_raw}")
    print(f"목표 최종 Q/A 수: {TARGET_FINAL_QA}")
    print(f"예상 필요 조문 수(단순 추정): {estimated_needed}")

    save_json(ARTICLE_JSON_PATH, articles)

    print("\n[조문 샘플 20개]")
    for a in articles[:20]:
        print(f"{a['article_id']} | {a['article_title']}")

    print("\n[3/5] Q/A 생성")
    raw_qa = generate_until_target(
        articles=articles,
        generation_config=GENERATION_CONFIG,
        target_final_qa=TARGET_FINAL_QA
    )
    save_json(RAW_QA_PATH, raw_qa)
    print(f"생성된 원본 Q/A 수: {len(raw_qa)}")

    print("[4/5] 중복 제거")
    deduped = deduplicate_qa(raw_qa)

    if len(deduped) > TARGET_FINAL_QA:
        deduped = deduped[:TARGET_FINAL_QA]

    save_json(DEDUP_QA_PATH, deduped)
    print(f"중복 제거 후 최종 Q/A 수: {len(deduped)}")

    print("[5/5] 파인튜닝 JSONL 저장")
    finetune_rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, finetune_rows)

    print("\n완료")
    print(f"- 조문 저장: {ARTICLE_JSON_PATH}")
    print(f"- 원본 Q/A: {RAW_QA_PATH}")
    print(f"- 중복 제거 Q/A: {DEDUP_QA_PATH}")
    print(f"- 파인튜닝 JSONL: {FINETUNE_JSONL_PATH}")

    if len(deduped) < TARGET_FINAL_QA:
        print(
            f"\n[주의] 목표 {TARGET_FINAL_QA}개보다 적은 {len(deduped)}개만 확보되었습니다. "
            "GENERATION_CONFIG를 늘리거나 MAX_PASSES를 올리세요."
        )


if __name__ == "__main__":
    main()

[1/5] MD 로드
[2/5] 조문 분리
전체 분리 조문 수: 217
실사용 조문 수: 217
조문당 요청 생성 수: 9
목표 최종 Q/A 수: 5000
예상 필요 조문 수(단순 추정): 715

[조문 샘플 20개]
C1.1 | Formula One World Championship
C1.2 | Regulatory Framework
C1.3 | Interpretation of and amendments to these Technical Regulations
C1.6 | New systems or technologies
C1.7 | Duty of Competitor and PU Manufacturer
C2.1 | Coordinate systems and conventions
C2.1.1 | Car Coordinate System
C2.1.2 | Further conventions
C2.1.3 | Wheel Coordinate System
C2.2 | Principal Planes
C2.3 | Fundamental Dimensions
C2.3.2 | Height
C2.3.3 | Wheelbase
C2.4 | Reference Volumes and Surfaces
C2.5 | Precision of Numerical Values
C3.1 | Aerodynamic Components or Bodywork
C3.1.1 | the groups specified in Articles C3.5 to C3.11.
C3.2 | General Principles
C3.2.1 | Objectives of Article C3
C3.2.2 | - a. Article C3.4.

[3/5] Q/A 생성

===== PASS 1/20 시작 =====
  - PASS 1 | (1/217) C16.11 Recycling of Engine Oil
  - PASS 1 | (2/217) C2.1.2 Further conventions
  - PASS 1 | (3/217) C3.7.5 Mirro